# Early Sepsis Prediction: Part 2, Novel Preprocessing

Builds the leakage-free BASE features and the novel feature families (P1 to P6) used by the split ablation in `03_split_ablation.ipynb`.

Needs `processed/sepsis_nodup.parquet` from notebook 01 (steps 1 and 2) in Drive. Every step saves a parquet checkpoint and loads it in later sessions. All features at hour t only use the patient's hours up to t.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_URL = "https://github.com/Thorfast191/Early-Sepsis-Detection-In-ICU-Patients.git"
REPO_DIR = "/content/Early-Sepsis-Detection-In-ICU-Patients"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

In [ ]:
%cd {REPO_DIR}
!pip install -q -r requirements.txt

In [ ]:
FORCE = False       # set True to recompute steps instead of loading checkpoints
QUICK_MODE = False  # True: run on a stratified sample of 4,000 patients (own checkpoint folders)

RUN_V2 = False      # True: also train the v2 arms (data fixes, site screening, group ablation)

os.environ["SEPSIS_QUICK_MODE"] = "1" if QUICK_MODE else "0"
os.environ["SEPSIS_RUN_V2"] = "1" if RUN_V2 else "0"

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.max_columns", 60)

In [ ]:
import importlib
from data_preprocessing import config, utils
from novel_preprocessing import (common, n00_observation_eda, n01_labels, n02_base_features,
                                 n03_observation_process, n04_patient_baseline, n05_clinical_scores,
                                 n06_trajectory, n07_fitted_transforms, n08_data_fixes, n09_site_screening)
from ablation import a01_make_splits

for module in [config, utils, common, n00_observation_eda, n01_labels, n02_base_features, n03_observation_process,
               n04_patient_baseline, n05_clinical_scores, n06_trajectory, n07_fitted_transforms,
               n08_data_fixes, n09_site_screening, a01_make_splits]:
    importlib.reload(module)

print("data folder:", config.PROJECT_DIR)
print("quick mode:", config.QUICK_MODE)

## 0. How often is each variable measured?

Before building anything we look at the *observation process*: how often each of the 34 vitals and labs is measured, in hospital A vs B and in septic vs non-septic patients, and how many hours pass between measurements. If the two hospitals measure very differently, a model that learns from raw measurement patterns may not carry over from one hospital to the other. That motivates the hospital-relative encoding (P2). A KS test compares the vital values of A and B.

In [ ]:
rates, gaps, ks = n00_observation_eda.run(force=FORCE)

**Observations:** (write after running)

## 1. Labels for several prediction horizons

In this dataset `SepsisLabel` turns to 1 six hours before clinical onset, so onset = first labelled hour + 6. For each horizon H (6, 12, 24, 48 h) a row is positive if onset is at most H hours ahead. Rows at or after onset are removed (they are trivially positive), and patients already labelled septic on their first ICU row are excluded (their onset happened before we see them). The same rows and labels are used by both arms.

In [ ]:
labels = n01_labels.run(force=FORCE)

**Observations:** (write after running)

## 2. BASE features

The basic pipeline of notebook 01 (the 14 columns kept by step 3, per-patient forward fill, ICU type, gender and age groups), but without anything learned from the data. The median fill and the scaler are fitted later on training patients only, inside every split.

In [ ]:
base = n02_base_features.run(force=FORCE)

**Observations:** (write after running)

## 3. P1: the observation process

For each of the 34 variables: the last measured value, the hours since it was last measured, whether it was ever measured, and how many times it was measured per ICU hour so far. A doctor ordering a lactate is itself a signal. The 26 labs dropped by step 3 come back here. EtCO2 only gets its timing features.

In [ ]:
p1 = n03_observation_process.run(force=FORCE)

**Observations:** (write after running)

## 4. P3: deviation from the patient's own baseline

A heart rate of 100 means different things for different people. For vitals the baseline is the median of the patient's first 6 hours (growing until hour 6, then fixed); for labs it is the first measured value. The feature is the current value minus that baseline.

In [ ]:
p3 = n04_patient_baseline.run(force=FORCE)

**Observations:** (write after running)

## 5. P4: clinical scores

Hourly scores doctors use: shock index, pulse pressure, BUN/creatinine, SaO2/FiO2 and SpO2/FiO2, a partial SOFA score (respiration, coagulation, liver, cardiovascular, renal) with a missing flag for each part, the rise in SOFA from the patient's lowest value so far (Sepsis-3 uses a rise of 2 or more), partial qSOFA and the SIRS count.

In [ ]:
p4 = n05_clinical_scores.run(force=FORCE)

**Observations:** (write after running)

## 6. P5: trajectories

Mean, standard deviation, minimum, maximum and slope over the last 6, 12 and 24 hours for the vitals, the shock index and SOFA. All windows only look back. The slope is the least-squares slope from rolling sums, so there is no Python loop over rows.

In [ ]:
p5 = n06_trajectory.run(force=FORCE)

**Observations:** (write after running)

## 7. Fitted transforms (P6 and P2)

These steps learn from data, so they are fitted inside every split on training patients only (the ablation notebook does this). Here we only show the time-decay constants.

- **P6 time-decay imputation:** an old measurement is less trustworthy. For each variable, tau is the lag (hours) at which its autocorrelation drops below 1/e. The imputed value moves from the last measurement towards the training median as exp(-hours since / tau).
- **P2 hospital-relative encoding:** values, hours-since and rates are turned into percentiles within the row's own hospital. For the external split the new hospital's reference can come from its own unlabeled data (transductive, no labels used).

This step also creates the patient-level splits (`processed/splits.parquet`) if they do not exist yet.

In [ ]:
tau = n07_fitted_transforms.run(force=FORCE)

**Observations:** (write after running)

## 8. v2 data fixes

Two variables mix two different measurements. Hospital B records ionized calcium (about 1.1 mmol/L) in the same column as total calcium (about 8.3 mg/dL), and hospital A has many SaO2 values below 80 that look like venous samples. Values below the cut-off become their own variable (`iCalcium`, `SvO2_suspect`); no unit conversion is assumed. The time-decay constant tau is now fitted as an exponential to the whole autocorrelation curve, so one early dip no longer decides it. This step builds the v2 checkpoints used by the v2 arms of the ablation.

In [ ]:
p1_v2, p3_v2, p4_v2 = n08_data_fixes.run(force=FORCE)

**Observations:** (write after running)